# 04 · Drift scenarios preview, and PSI by hand

**Who this is for:** you, after notebook 03b.
**You should already know:** physiology vs measurement (02), the truth generator (03), hospital activity (03b).

**By the end you will be able to:**
- run an **episode** with any of the four drift scenarios switched on
- compute the **Population Stability Index (PSI)** step by step, and the Jensen-Shannon distance
- explain why **KS-test p-values** are a poor drift alarm on large data
- say which monitor catches each scenario, and what the **right response** is (fix data vs retrain)

**Outline**
1. Setup
2. The four scenarios
3. A normal episode (the reference)
4. PSI by hand
5. Why not KS p-values?
6. Scenario 1: SpO2 sensor bias (data drift)
7. Scenario 2: respiratory outbreak (abrupt drift)
8. Scenario 3: protocol change (concept drift)
9. Scenario 4: firmware schema change (schema drift)
10. Which monitor catches what
11. Exercise
12. Pitfalls and next steps

Code used: `generator/drift.py`, `generator/drift_config.yaml`, `databricks/src/heavden/monitoring/drift_metrics.py`.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from generator import drift, hospital, synthea
from heavden.monitoring import drift_metrics


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()
csv_dir = synthea.run_synthea(RUN, ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}")
profiles = synthea.build_profiles(
    synthea.load_tables(csv_dir), pd.Timestamp(RUN.reference_date, tz="UTC")
)
inpatients = hospital.admit_inpatients(profiles)
START = hospital.SIM_START
HOURS = 48
SEED = 42


def episode(*scenarios):
    return drift.simulate_episode(profiles, inpatients, START, HOURS, scenarios, seed=SEED)

## 2. The four scenarios

They're defined in `generator/drift_config.yaml`. Each one pulls a **different lever** in the simulator, so each one tests a different part of the platform:

In [ ]:
config = drift.load_config()
pd.DataFrame(config).T.fillna("")

| Scenario | Lever | What it's like in real life |
|---|---|---|
| `spo2_sensor_bias` | **measurement** offsets on Site B devices | A firmware bug makes oximeters read low. Patients are fine. |
| `respiratory_outbreak` | **who gets admitted** | Flu/pneumonia season: a new kind of patient fills the wards |
| `protocol_change` | the **truth** rule | A new escalation protocol: staff call for help earlier |
| `firmware_schema_change` | the **message format** | Devices start sending `etco2` and stop sending `motion` |

Start times are **hours into the episode** (e.g. 12 h), so the same config works for any episode.

## 3. A normal episode (the reference)

`drift.simulate_episode` runs the hospital for 48 simulated hours: activity, vitals, truth.
With the **same seed**, a scenario episode is identical to the normal one **until its scenario starts**. Any difference afterwards is caused by the scenario.

In [ ]:
%%time
normal = episode()
print(f"{len(normal.readings):,} readings | {len(normal.ward.truth.outcomes)} escalations")

In [ ]:
def window(readings, from_hour, to_hour, site=None):
    """Readings between two episode hours, optionally for one site."""
    t0, t1 = START + pd.Timedelta(hours=from_hour), START + pd.Timedelta(hours=to_hour)
    r = readings[(readings["ts"] >= t0) & (readings["ts"] < t1)]
    if site:
        r = r[r["device_id"].str.startswith(f"DEV-{site[-1]}")]
    return r

## 4. PSI by hand

**PSI (Population Stability Index)** answers: *how differently is this variable distributed now, compared with a baseline?*

The recipe:
1. Cut the **baseline** into 10 bins at its deciles, so each bin holds ~10% of the baseline.
2. Count what share of the **current** data falls in each bin.
3. For each bin: `(actual − expected) × ln(actual / expected)`.
4. Add them up. **< 0.1 stable · 0.1–0.25 moderate shift (warn) · > 0.25 major shift (act).**

Let's do it on **Site B SpO2, hours 24–48**: normal vs sensor-biased episode.

In [ ]:
biased = episode("spo2_sensor_bias")
baseline = window(normal.readings, 24, 48, "SITE_B")["spo2"].to_numpy()
current = window(biased.readings, 24, 48, "SITE_B")["spo2"].to_numpy()

# Step 1: bins from the baseline deciles (open-ended at both ends)
edges = drift_metrics.baseline_bins(baseline, n_bins=10)
print("bin edges:", edges)

SpO2 is a whole number, so several deciles coincide. Duplicate edges are merged, and that's why there are fewer than 10 bins. That's fine.

In [ ]:
# Steps 2-3: shares per bin and each bin's contribution
expected = drift_metrics.bin_shares(baseline, edges)
actual = drift_metrics.bin_shares(current, edges)
eps = 1e-4
contribution = (np.clip(actual, eps, None) - np.clip(expected, eps, None)) * np.log(
    np.clip(actual, eps, None) / np.clip(expected, eps, None)
)
labels = [f"({lo:g}, {hi:g}]" for lo, hi in zip(edges[:-1], edges[1:], strict=True)]
table = pd.DataFrame(
    {"expected": expected, "actual": actual, "contribution": contribution}, index=labels
).round(4)
table

In [ ]:
# Step 4: add up
psi_by_hand = table["contribution"].sum()
psi_library = drift_metrics.psi(baseline, current)
level = drift_metrics.psi_level(psi_library)
print(f"PSI by hand: {psi_by_hand:.3f} | heavden.monitoring: {psi_library:.3f} -> {level}")
js = drift_metrics.js_distance(baseline, current)
print(f"JS distance: {js:.3f}  (0 = identical, 1 = no overlap)")

**Interpretation:** the biased episode has fewer readings in the high bins and more in the low bins. Each bin's contribution is positive, and together they give the PSI above, compared against the 0.1 (warn) and 0.25 (act) thresholds.

The library version (used later in Databricks) gives the same number. It also refuses to judge tiny samples (returns *NaN* under 500 values).

## 5. Why not KS p-values?

The **Kolmogorov–Smirnov test** asks *"could these two samples come from the same distribution?"* and gives a p-value.
The problem: with enough data, **any** difference, however tiny, becomes "significant".

Let's shift heart rate by a clinically meaningless **0.3 bpm** and grow the sample:

In [ ]:
rng = np.random.default_rng(0)
hr = normal.readings["heart_rate"].to_numpy().astype(float)
rows = []
for n in [1_000, 10_000, 100_000, 400_000]:
    a = rng.choice(hr, n)
    b = rng.choice(hr, n) + 0.3 + rng.normal(0, 0.5, n)  # tiny shift (+ jitter to break ties)
    rows.append(
        {
            "sample size": f"{n:,}",
            "KS p-value": stats.ks_2samp(a, b).pvalue,
            "PSI": drift_metrics.psi(a, b),
        }
    )
ks_table = pd.DataFrame(rows)
ks_table["KS p-value"] = ks_table["KS p-value"].map("{:.1e}".format)
ks_table["PSI"] = ks_table["PSI"].round(4)
ks_table

**Interpretation:** the KS p-value collapses toward zero as the sample grows, so it would alarm constantly on a hospital producing ~85,000 readings a day.
PSI measures the **size** of the shift, so it stays near zero for a meaningless change regardless of volume. That's why the platform uses PSI/JS with a minimum-sample guard.

## 6. Scenario 1: SpO2 sensor bias (data drift)

From hour 12, Site B devices read SpO2 lower and lower, by 1.5 points per day, capped at 3.

In [ ]:
def hourly_mean(readings, column, site=None):
    r = (
        readings
        if site is None
        else readings[readings["device_id"].str.startswith(f"DEV-{site[-1]}")]
    )
    return r.groupby(r["ts"].dt.floor("h"))[column].mean()


fig, ax = plt.subplots(figsize=(10, 3))
for name, ep in {"normal": normal, "sensor bias": biased}.items():
    hourly_mean(ep.readings, "spo2", "SITE_B").plot(ax=ax, label=name)
ax.axvline(START + pd.Timedelta(hours=12), color="red", ls="--", lw=1)
ax.set_title("Site B: average SpO2 reported each hour")
ax.legend();

In [ ]:
rows = []
for h0 in range(0, HOURS, 12):
    row = {"window": f"{h0}-{h0 + 12}h"}
    for site in ["SITE_A", "SITE_B", "SITE_C"]:
        row[site] = drift_metrics.psi(
            window(normal.readings, h0, h0 + 12, site)["spo2"],
            window(biased.readings, h0, h0 + 12, site)["spo2"],
        )
    row["all sites"] = drift_metrics.psi(
        window(normal.readings, h0, h0 + 12)["spo2"], window(biased.readings, h0, h0 + 12)["spo2"]
    )
    rows.append(row)
pd.DataFrame(rows).set_index("window").round(3)

**Interpretation:**
- PSI rises **only at Site B**, and grows with time (the bias keeps growing).
- The **all-sites** PSI is much weaker, because the other two sites dilute it. That's why monitors are **sliced by site**.
- The patients' **true** SpO2 didn't change (notebook 02 §10), so the right response is to **fix the sensors, not retrain**. A retrained model would learn the bug.

## 7. Scenario 2: respiratory outbreak (abrupt drift)

From hour 12, ~60% of new admissions arrive with pneumonia: faster breathing, fever, lower oxygen, and a higher risk of deteriorating.

In [ ]:
outbreak = episode("respiratory_outbreak")
enc = outbreak.plan.encounters
new = enc[enc["admit_ts"] >= START]
n_pneumonia = (new["admission_reason"] == "pneumonia").sum()
print(f"admissions in the episode: {len(new)} | with pneumonia: {n_pneumonia}")
n_normal, n_outbreak = len(normal.ward.truth.outcomes), len(outbreak.ward.truth.outcomes)
print(f"escalations: normal {n_normal} -> outbreak {n_outbreak}")

In [ ]:
vitals_to_check = ["resp_rate", "temp_c", "spo2", "heart_rate"]
rows = []
for h0 in range(0, HOURS, 12):
    rows.append(
        {"window": f"{h0}-{h0 + 12}h"}
        | {
            v: drift_metrics.psi(
                window(normal.readings, h0, h0 + 12)[v], window(outbreak.readings, h0, h0 + 12)[v]
            )
            for v in vitals_to_check
        }
    )
pd.DataFrame(rows).set_index("window").round(3)

**Interpretation:** PSI grows window by window as more pneumonia patients are admitted. Breathing rate and temperature move most.
Unlike the sensor bias, these patients **really are** different and sicker. If the model starts mis-predicting for them, **retraining** on the new mix is justified.

## 8. Scenario 3: protocol change (concept drift)

From hour 12, a new protocol does two things:
- staff escalate deteriorating patients **earlier** (at 70% of a trajectory instead of 100%)
- staff **review** stable patients whose vitals are persistently abnormal (warning score ≥ 1), escalating about 10% of them each hour

The **inputs don't change at all**, only what happens to patients with those inputs.

In [ ]:
protocol = episode("protocol_change")
rows = []
for h0 in range(0, HOURS, 12):
    rows.append(
        {"window": f"{h0}-{h0 + 12}h"}
        | {
            v: drift_metrics.psi(
                window(normal.readings, h0, h0 + 12)[v], window(protocol.readings, h0, h0 + 12)[v]
            )
            for v in ["spo2", "resp_rate", "heart_rate"]
        }
    )
psi_table = pd.DataFrame(rows).set_index("window").round(3)


def escalations_per_window(ep):
    ev = ep.ward.truth.outcomes["event_ts"]
    return [
        int(
            (
                (ev >= START + pd.Timedelta(hours=h)) & (ev < START + pd.Timedelta(hours=h + 12))
            ).sum()
        )
        for h in range(0, HOURS, 12)
    ]


psi_table["escalations (normal)"] = escalations_per_window(normal)
psi_table["escalations (new protocol)"] = escalations_per_window(protocol)
print(
    protocol.ward.truth.trajectories.query("outcome == 'escalated'")["kind"]
    .value_counts()
    .to_dict()
)
psi_table

**Interpretation:** the vitals' PSI stays small (the only differences are patients who left earlier), yet escalations rise after hour 12.
An input-drift monitor would **miss this**. Only comparing predictions with the **delayed labels** reveals it: the model's accuracy falls once outcomes arrive.
That's why episode 2 (day 8) is detected by **performance monitoring**, and why it needs labels, so it takes time.

## 9. Scenario 4: firmware schema change (schema drift)

From hour 6, devices run firmware 3.2.0: they add `etco2` (exhaled CO2) and stop sending `motion`.

In [ ]:
firmware = episode("firmware_schema_change")
r = firmware.readings
by_hour = r.groupby(r["ts"].dt.floor("2h")).agg(
    firmware=("firmware", lambda s: ", ".join(sorted(s.unique()))),
    motion_missing=("motion", lambda s: f"{s.isna().mean():.0%}"),
    etco2_present=("etco2", lambda s: f"{s.notna().mean():.0%}"),
)
by_hour.head(6)

In [ ]:
def as_json(row):
    """What the device actually sends: missing fields are simply absent."""
    return {k: v for k, v in row.items() if not (isinstance(v, float) and np.isnan(v))}


before = r[r["ts"] < START + pd.Timedelta(hours=6)].iloc[0]
after = r[r["ts"] >= START + pd.Timedelta(hours=6)].iloc[0]
print("before:", sorted(as_json(before.to_dict())))
print("after: ", sorted(as_json(after.to_dict())))

**Interpretation:** this isn't a statistical shift, it's a **contract** change. It's caught at ingestion:
- Auto Loader's **schema evolution** adds the new `etco2` column (or puts it in `_rescued_data`)
- a **data-quality expectation** on `motion` (e.g. "null rate < 5%") fails
- the **schema-change log** records firmware 3.2.0

The right response is to update the pipeline and features to the new contract (and decide whether `etco2` becomes a feature), **not** to retrain on broken inputs.

## 10. Which monitor catches what

| Scenario | Input PSI | Performance (needs labels) | Data quality / schema | Right response |
|---|---|---|---|---|
| SpO2 sensor bias | ✅ Site B slice only | maybe, slowly | – | **Fix sensors, don't retrain** |
| Respiratory outbreak | ✅ breathing, temperature | ✅ if the model under-predicts | – | **Retrain** on the new mix (after review) |
| Protocol change | ❌ stays quiet | ✅ the **only** signal | – | **Retrain** (outcomes changed) |
| Firmware schema change | – | – | ✅ immediately | **Update the pipeline contract** |

This table is the heart of the case study: **a good MLOps system doesn't just detect drift, it helps a human choose the right response.**

## 11. Exercise: when does the outbreak cross the PSI thresholds?

Using `normal` and `outbreak`:
1. compute the PSI of **`resp_rate`** in **6-hour** windows (0–6 h, 6–12 h, …, 42–48 h)
2. find the first window where PSI ≥ 0.1 (**warn**) and the first where PSI ≥ 0.25 (**act**)
3. bonus: repeat for Site A only. Is it faster or slower than all sites together?

In [ ]:
# Try it yourself
psi_by_window = None  # dict or Series: window -> PSI

first_warn = None
first_act = None

In [ ]:
# Answer (run after trying)
psi_by_window = pd.Series(
    {
        f"{h}-{h + 6}h": drift_metrics.psi(
            window(normal.readings, h, h + 6)["resp_rate"],
            window(outbreak.readings, h, h + 6)["resp_rate"],
        )
        for h in range(0, HOURS, 6)
    }
).round(3)
first_warn = psi_by_window[psi_by_window >= drift_metrics.PSI_WARN].index.min()
first_act = psi_by_window[psi_by_window >= drift_metrics.PSI_ACT].index.min()
print("first warn window:", first_warn, "| first act window:", first_act)
psi_by_window

The outbreak builds gradually because only *new admissions* have pneumonia, so the ward changes as beds turn over. It might not reach **act** within 48 hours at all.
That's realistic, and it's why the monitoring compares windows over **days**, not hours.

## 12. Pitfalls and next steps

**Pitfall: a baseline from a different time of day.** Vitals follow a day/night rhythm (notebook 02). Comparing a night window with a daytime baseline shows "drift" that's just the clock. Here we compared the **same** hours of two episodes. In production the baseline is the champion's training data, covering many full days.

**Pitfall: only looking at the whole hospital.** The sensor bias was obvious at Site B and diluted overall. Always slice monitors by site (and unit).

**Pitfall: retraining on every alert.** Retraining on a sensor bug or a schema break bakes the problem into the model. Diagnose first: *did the patients change, or did the data?*

**Pitfall: too little data.** PSI on 50 readings is noise. The library returns NaN under 500 values, a deliberate guard.

**Optional extension:** run `episode("spo2_sensor_bias", "respiratory_outbreak")` and check whether you can still tell the two apart by slicing (by site, and by vital).

**Next notebook:** `05_features_and_labels_local`. Turn readings into hourly features, join them to 6-hour labels point-in-time, and spot leakage on purpose.